# Cookie Cats A/B Test — Phase 1: Exploratory Data Analysis & Cleaning

## 1. Project Background
Cookie Cats is a classic mobile puzzle game developed by Tactile Entertainment. As players progress through the game, they encounter **gates** that force them to wait a certain time or make an in-app purchase before progressing.

In this A/B test:
- **Control (`gate_30`)**: Gate placed at Level 30.
- **Treatment (`gate_40`)**: Gate moved to Level 40.

### Primary Objective
Analyze whether shifting the gate from level 30 to level 40 impacts player retention (Day-1 and Day-7) and player engagement (total game rounds played in the first 14 days).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set styling for publication-quality charts
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

## 2. Data Loading & Initial Inspection

In [ ]:
from src.data_prep import load_and_clean

raw_df = pd.read_csv('../data/cookie_cats.csv')
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

In [ ]:
# Schema and null checks
print("Missing values count per column:")
print(raw_df.isnull().sum())
print("\nData Types:")
print(raw_df.dtypes)

## 3. Outlier Identification & Data Cleaning
Let's inspect the upper tail of `sum_gamerounds` to detect anomalous entries.

In [ ]:
# Inspect top 10 players by gamerounds
raw_df.sort_values(by='sum_gamerounds', ascending=False).head(10)

> **Observation**: User ID `6390605` recorded **49,854** game rounds in 14 days (~3,561 rounds/day, which is physically impossible for normal human play or represents an automated bot/telemetry error). The next highest player played **2,961** rounds. We drop this single extreme outlier.

In [ ]:
df = load_and_clean('../data/cookie_cats.csv')
print(f"Cleaned dataset shape: {df.shape}")

## 4. Sample Split & Balance Check

In [ ]:
group_counts = df['version'].value_counts()
group_pcts = df['version'].value_counts(normalize=True) * 100

split_summary = pd.DataFrame({
    'User Count': group_counts,
    'Percentage (%)': group_pcts.round(2)
})
split_summary

## 5. Distribution of Engagement (`sum_gamerounds`)
Examining the distribution of game rounds played to evaluate normality assumptions.

In [ ]:
skewness = df['sum_gamerounds'].skew()
kurt = df['sum_gamerounds'].kurtosis()
print(f"Skewness of sum_gamerounds: {skewness:.2f} (Extremely right-skewed)")
print(f"Kurtosis of sum_gamerounds: {kurt:.2f}")

# Summary statistics by group
df.groupby('version')['sum_gamerounds'].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.99])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of rounds up to 99th percentile
p99 = df['sum_gamerounds'].quantile(0.99)
sns.histplot(data=df[df['sum_gamerounds'] <= p99], x='sum_gamerounds', hue='version', 
             kde=True, bins=40, ax=axes[0], alpha=0.4)
axes[0].set_title('Distribution of sum_gamerounds (<= 99th Percentile)')
axes[0].set_xlabel('Game Rounds Played')
axes[0].set_ylabel('Player Count')

# Log-scale distribution
sns.boxplot(data=df, x='version', y='sum_gamerounds', ax=axes[1], palette='Set2')
axes[1].set_yscale('log')
axes[1].set_title('Log-Scale Boxplot of sum_gamerounds by Group')
axes[1].set_ylabel('Game Rounds (Log Scale)')
axes[1].set_xlabel('A/B Test Version')

plt.tight_layout()
plt.show()

> **Methodological Implication**: Because `sum_gamerounds` is heavily right-skewed and non-normal (over 50% of players play fewer than 20 rounds), comparing groups using a standard two-sample Student's t-test violates standard normality assumptions. Therefore, we must use the **non-parametric Mann-Whitney U test** for engagement testing in Phase 3.

## 6. Retention Rate Overview (Day-1 and Day-7)

In [ ]:
retention_summary = df.groupby('version').agg(
    n_users=('userid', 'count'),
    retention_1_rate=('retention_1', 'mean'),
    retention_7_rate=('retention_7', 'mean')
).reset_index()

retention_summary['retention_1_pct'] = (retention_summary['retention_1_rate'] * 100).round(2)
retention_summary['retention_7_pct'] = (retention_summary['retention_7_rate'] * 100).round(2)
retention_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Day 1 Retention Plot
sns.barplot(data=df, x='version', y='retention_1', ax=axes[0], ci=95, capsize=0.1, palette='Blues_d')
axes[0].set_title('Day-1 Retention Rate by Version (with 95% CI)')
axes[0].set_ylabel('Retention Rate')
axes[0].set_ylim(0.40, 0.48)

# Day 7 Retention Plot
sns.barplot(data=df, x='version', y='retention_7', ax=axes[1], ci=95, capsize=0.1, palette='Greens_d')
axes[1].set_title('Day-7 Retention Rate by Version (with 95% CI)')
axes[1].set_ylabel('Retention Rate')
axes[1].set_ylim(0.15, 0.22)

plt.tight_layout()
plt.show()

## 7. Key EDA Takeaways
1. **Clean Baseline**: Dropped 1 unphysical outlier (`userid: 6390605`, 49,854 rounds), leaving 90,188 validated records.
2. **Group Sizes**: `gate_30` has 44,699 users (49.56%), `gate_40` has 45,489 users (50.44%).
3. **Engagement Skew**: Median rounds played is 16 (mean ~51), confirming heavy right skew and mandating non-parametric testing (Mann-Whitney U).
4. **Initial Retention Signal**:
   - Day-1 Retention: `gate_30` = 44.82% vs `gate_40` = 44.23% (Δ = -0.59 percentage points)
   - Day-7 Retention: `gate_30` = 19.02% vs `gate_40` = 18.20% (Δ = -0.82 percentage points)
   - Formal statistical significance will be evaluated in Phase 3.